In [99]:
import modal

#here I'm basically just launching a Modal app and definint the volume etc. (I've already uploaded the data. Commented-out code)

app = modal.App("leaf-disease-cnn")

image = (
    modal.Image.debian_slim()
    .pip_install(
        "torch", "torchvision", "numpy", "pandas",
        "scikit-learn", "Pillow", "tqdm", "optuna"
    )
    .add_local_file("dataset_and_CNN_utils.py", remote_path = "/root/dataset_and_CNN_utils.py")
)

#data's in data_vol, models will be saved in ckpt_vol
data_vol = modal.Volume.from_name("plant-pictures", create_if_missing=True)
ckpt_vol = modal.Volume.from_name("plant-models", create_if_missing=True)

# Mount points used inside Modal containers
DATA_ROOT = "/data/data"       # volume mount /data + internal folder data
CKPT_ROOT = "/ckpt"

# old code to upload data to the volume
# from pathlib import Path
# DATA_DIR = Path('C:/Users/franc/.cache/kagglehub/competitions/plant-pathology-2021-fgvc8')
# with data_vol.batch_upload() as batch:
#     batch.put_directory(DATA_DIR, "/data")

Prepare data for later processing (train val split)

In [100]:
@app.function(image=image, volumes={"/data": data_vol, "/ckpt": ckpt_vol})
def prepare_data():
    import json
    from pathlib import Path
    import pandas as pd
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MultiLabelBinarizer

    # Make sure Modal sees the latest files uploaded to the volume.
    try:
        data_vol.reload()
    except Exception:
        pass

    data_root = Path("/data/data")
    train_csv = data_root / "train.csv"
    image_dir = data_root / "train_images"

    if not (train_csv.exists() and image_dir.exists()):
        raise FileNotFoundError(
            f"Could not find {train_csv} or {image_dir}"
        )

    df = pd.read_csv(train_csv)
    df["labels"] = df["labels"]
    df["label_list"] = df["labels"].apply(lambda x: str(x).split())

    mlb = MultiLabelBinarizer()
    mlb.fit(df["label_list"])

    train_df, val_df = train_test_split(
        df,
        test_size=0.15,
        random_state=42,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(data_root / "train_split.csv", index=False)
    val_df.to_csv(data_root / "val_split.csv", index=False)

    with open(data_root / "classes.json", "w") as f:
        json.dump(list(mlb.classes_), f, indent=2)

    # Also save a small metadata file for quick inspection.
    meta = {
        "num_train": int(len(train_df)),
        "num_val": int(len(val_df)),
        "classes": list(mlb.classes_),
        "num_classes": int(len(mlb.classes_)),
    }
    with open(data_root / "dataset_metadata.json", "w") as f:
        json.dump(meta, f, indent=2)

    try:
        data_vol.commit()
    except Exception:
        pass

    return meta


Instantiated data loaders, define a flexible CNN, then train it as a baseline for the rest of the experiments

In [101]:
@app.function(
    image=image,
    volumes={"/data": data_vol, "/ckpt": ckpt_vol},
    timeout=60 * 60 * 4,
    gpu="L40S",
)
def train_cnn(num_epochs=10, batch_size=32, lr=1e-4, image_size=224, num_workers=8, prefetch_factor=2):
    import json
    from pathlib import Path
    from dataset_and_CNN_utils import make_data_loader, FlexibleCNN

    import pandas as pd
    import torch
    import torch.nn as nn

    
    #reload volumes to make sure we see the latest data and checkpoints
    try:
        data_vol.reload()
        ckpt_vol.reload()
    except Exception:
        pass
    
    #usual redefinition of paths
    data_root = Path("/data/data")
    image_dir = data_root / "train_images"
    train_csv = data_root / "train_split.csv"
    val_csv = data_root / "val_split.csv"

    # 1. Get again the label_list and mlb
    classes_path = data_root / "classes.json"

    with open(classes_path, "r") as f:
        classes = json.load(f)
    num_classes = len(classes)


    # 2. Detect image column
    
    image_col = "image"
    label_col = "labels"
    print(f"Number of classes: {num_classes}")
    print(f"Classes: {classes}")

    # 3. DataLoaders

    train_loader = make_data_loader(
        csv_file=train_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        prefetch_factor=prefetch_factor,
        shuffle=True,
    )

    val_loader = make_data_loader(
        csv_file=val_csv,
        image_dir=image_dir,
        classes=classes,
        batch_size=batch_size,
        image_size=image_size,
        num_workers=num_workers,
        prefetch_factor=prefetch_factor,
        shuffle=False,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = FlexibleCNN(
        num_layers=3, 
        num_filters=32,
        kernel_size=3,
        num_classes=num_classes,
        batch_norm_included=False,
        image_size=image_size,
        dropout_rate=0.5,
    ).to(device)

    # 4. Loss and optimizer
    
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    # 5. Training and validation loops
    
    history = []

    best_val_loss = float("inf")
    ckpt_dir = Path("/ckpt")
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_model_path = ckpt_dir / "best_simple_cnn.pt"
    last_model_path = ckpt_dir / "last_simple_cnn.pt"
    history_path = ckpt_dir / "training_history.csv"

    for epoch in range(num_epochs):
        print(f"Starting epoch {epoch + 1}/{num_epochs}", flush=True)

        model.train()

        train_loss_sum = 0.0
        train_num_samples = 0

        for batch_idx, (images, targets) in enumerate(train_loader):
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, targets)

            loss.backward()
            optimizer.step()

            batch_size_actual = images.size(0)
            train_loss_sum += loss.item() * batch_size_actual
            train_num_samples += batch_size_actual

            if batch_idx % 20 == 0:
                print(
                    f"Epoch {epoch + 1}/{num_epochs} | "
                    f"Batch {batch_idx}/{len(train_loader)} | "
                    f"loss={loss.item():.4f}",
                    flush=True,
                )

        train_loss = train_loss_sum / train_num_samples

        # Validation
        model.eval()

        val_loss_sum = 0.0
        val_num_samples = 0

        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                targets = targets.to(device)

                logits = model(images)

                loss = criterion(logits, targets)

                batch_size_actual = images.size(0)
                val_loss_sum += loss.item() * batch_size_actual
                val_num_samples += batch_size_actual

        val_loss = val_loss_sum / val_num_samples

        row = {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
        }

        history.append(row)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f}"
        )

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "classes": classes,
                    "num_classes": num_classes,
                    "image_size": image_size,
                    "epoch": epoch + 1,
                    "val_loss": val_loss,
                },
                best_model_path,
            )

            print(f"Saved new best model to {best_model_path}")

    # 6. Save final model and training history
    
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "classes": classes,
            "num_classes": num_classes,
            "image_size": image_size,
            "epoch": num_epochs,
            "val_loss": val_loss,
        },
        last_model_path,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    try:
        ckpt_vol.commit()
    except Exception:
        pass

    return {
        "num_epochs": num_epochs,
        "best_val_loss": best_val_loss,
        "best_model_path": str(best_model_path),
        "last_model_path": str(last_model_path),
        "history_df": history_df,
        "history_path": str(history_path),
        "num_classes": num_classes,
        "classes": classes,
    }


In [102]:
if __name__ == "__main__": 
    with app.run():
        metadata = prepare_data.remote()
        print("Dataset metadata:", metadata)

        train_results = train_cnn.remote(
            num_epochs=3,
            batch_size=64,
            lr=1e-4,
            image_size=224,
            num_workers=8,
            prefetch_factor=2,
        )
        print("Training results:", train_results)

Dataset metadata: {'num_train': 15837, 'num_val': 2795, 'classes': ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab'], 'num_classes': 6}
Training results: {'num_epochs': 3, 'best_val_loss': 0.3668214837852232, 'best_model_path': '/ckpt/best_simple_cnn.pt', 'last_model_path': '/ckpt/last_simple_cnn.pt', 'history_df':    epoch  train_loss  val_loss
0      1    0.429662  0.393618
1      2    0.380263  0.376052
2      3    0.368064  0.366821, 'history_path': '/ckpt/training_history.csv', 'num_classes': 6, 'classes': ['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab']}
